# Dashboard COVID-19 Jawa Barat

Notebook ini menjalankan `app.py` menggunakan Streamlit. Kode dan tampilan dashboard tetap berada di `app.py`, jadi hasilnya sama dengan saat dijalankan dari terminal.

Jalankan sel pertama untuk memulai dashboard, lalu buka tautan lokal yang muncul. Setelah selesai, jalankan sel **Hentikan dashboard**.

In [17]:
from pathlib import Path
import os
import socket
import subprocess
import tempfile
import time
from urllib.error import URLError
from urllib.request import urlopen

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "app.py").is_file():
    PROJECT_DIR = Path(r"d:\\Klompok4\\covid_dashboard_project\\covid_dashboard")

APP_FILE = PROJECT_DIR / "app.py"
PYTHON = PROJECT_DIR / ".venv" / "Scripts" / "python.exe"

if not APP_FILE.is_file():
    raise FileNotFoundError(f"app.py tidak ditemukan di {PROJECT_DIR}. Buka notebook dari folder proyek dashboard.")
if not PYTHON.is_file():
    raise FileNotFoundError(f"Python environment proyek tidak ditemukan: {PYTHON}. Buat .venv dan instal requirements.txt terlebih dahulu.")

if "dashboard_process" not in globals() or dashboard_process is None or dashboard_process.poll() is not None:
    port_socket = socket.socket()
    port_socket.bind(("127.0.0.1", 0))
    DASHBOARD_PORT = port_socket.getsockname()[1]
    port_socket.close()

    dashboard_log = tempfile.NamedTemporaryFile(prefix="covid-dashboard-", suffix=".log", delete=False)
    dashboard_log_path = Path(dashboard_log.name)
    dashboard_log.close()
    dashboard_log_handle = dashboard_log_path.open("w", encoding="utf-8")

    creation_flags = subprocess.CREATE_NEW_PROCESS_GROUP if os.name == "nt" else 0
    dashboard_process = subprocess.Popen(
        [str(PYTHON), "-m", "streamlit", "run", str(APP_FILE), "--server.headless", "true", "--server.port", str(DASHBOARD_PORT)],
        cwd=PROJECT_DIR,
        stdout=dashboard_log_handle,
        stderr=subprocess.STDOUT,
        creationflags=creation_flags,
    )
    DASHBOARD_URL = f"http://localhost:{DASHBOARD_PORT}"

    for _ in range(30):
        if dashboard_process.poll() is not None:
            dashboard_log_handle.close()
            raise RuntimeError(f"Streamlit berhenti saat mulai. Log:\n{dashboard_log_path.read_text(encoding='utf-8', errors='replace')}")
        try:
            with urlopen(DASHBOARD_URL, timeout=1) as response:
                if response.status == 200:
                    break
        except (URLError, TimeoutError, OSError):
            time.sleep(1)
    else:
        dashboard_process.terminate()
        dashboard_process.wait(timeout=10)
        dashboard_log_handle.close()
        raise TimeoutError(f"Dashboard belum merespons. Log:\n{dashboard_log_path.read_text(encoding='utf-8', errors='replace')}")

    print(f"Dashboard berjalan: {DASHBOARD_URL}")
else:
    print(f"Dashboard sudah berjalan: {DASHBOARD_URL}")

Dashboard sudah berjalan: http://localhost:52070


## Hentikan dashboard

Jalankan sel berikut saat selesai menggunakan dashboard.

In [15]:
if "dashboard_process" in globals() and dashboard_process is not None and dashboard_process.poll() is None:
    dashboard_process.terminate()
    try:
        dashboard_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        dashboard_process.kill()
        dashboard_process.wait(timeout=5)
    if "dashboard_log_handle" in globals():
        dashboard_log_handle.close()
    if "dashboard_log_path" in globals():
        print(f"Dashboard sudah dihentikan. Log startup tersimpan di: {dashboard_log_path}")
    else:
        print("Dashboard sudah dihentikan.")
else:
    print("Dashboard tidak sedang berjalan.")

Dashboard sudah dihentikan. Log startup tersimpan di: C:\Users\USER\AppData\Local\Temp\covid-dashboard-g2cxu_v_.log
